In [ ]:
"""
RAG 기반 임상시험 프로토콜 생성기
- FAISS에서 유사 임상시험 검색
- 검색된 실제 데이터를 Few-shot 예시로 활용
- LLaMA 3.1로 프로토콜 생성
"""

import os
import json
import time
import pickle
import random
import numpy as np
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline, set_seed
import faiss

# ============================================================
# 1. 설정
# ============================================================

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# 경로 설정
INDEX_PATH = "faiss_index"

# 모델 설정
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"  # 또는 Qwen3-Embedding
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

# GPU 설정
EMBEDDING_DEVICE = "cuda:0"  # 임베딩 모델용 GPU
LLM_DEVICE = "cuda:1"        # LLM용 GPU

# 검색 설정
TOP_K = 3  # 검색할 유사 임상시험 개수

# Seed for reproducibility
SEED = 42

# 2. Seed Fixing Function
# ============================================================
def set_all_seeds(seed: int = SEED):
    """Fix all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)  # Hugging Face transformers seed
    
    # For CUDA deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    print(f"✅ All seeds fixed to {seed}")

# ============================================================
# 2. 인덱스 및 모델 로딩
# ============================================================

class ClinicalTrialRAG:
    def __init__(self):
        self.index = None
        self.nct_ids = None
        self.study_data = None
        self.embed_tokenizer = None
        self.embed_model = None
        self.llm_pipe = None
        self.llm_tokenizer = None
        self.llm_model = None
    
    def load_faiss_index(self):
        """FAISS 인덱스 및 메타데이터 로딩"""
        start = time.time()
        print("🔄 FAISS 인덱스 로딩 중...")
        
        self.index = faiss.read_index(os.path.join(INDEX_PATH, "clinical_trials.index"))
        
        with open(os.path.join(INDEX_PATH, "nct_ids.pkl"), 'rb') as f:
            self.nct_ids = pickle.load(f)
        
        with open(os.path.join(INDEX_PATH, "study_data.pkl"), 'rb') as f:
            self.study_data = pickle.load(f)
        
        print(f"✅ FAISS 인덱스 로딩 완료: {self.index.ntotal:,}개 ({time.time() - start:.1f}초)")
    
    def load_embedding_model(self):
        """임베딩 모델 로딩"""
        start = time.time()
        print("🔄 임베딩 모델 로딩 중...")
        
        self.embed_tokenizer = AutoTokenizer.from_pretrained(
            EMBEDDING_MODEL, 
            trust_remote_code=True
        )
        self.embed_model = AutoModel.from_pretrained(
            EMBEDDING_MODEL,
            torch_dtype=torch.float16,
            trust_remote_code=True
        ).to(EMBEDDING_DEVICE)
        self.embed_model.eval()
        
        print(f"✅ 임베딩 모델 로딩 완료 ({time.time() - start:.1f}초)")
    
    def load_llm(self):
        """LLM 로딩"""
        start = time.time()
        print("🔄 LLM 로딩 중...")
        
        self.llm_tokenizer = AutoTokenizer.from_pretrained(
            LLM_MODEL,
            token=HF_TOKEN,
            trust_remote_code=True
        )
        
        # 수정: int로 변환
        gpu_id = int(LLM_DEVICE.split(":")[-1])
        
        self.llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL,
            token=HF_TOKEN,
            torch_dtype=torch.float16,
            device_map={"": gpu_id},  # int로 전달
            trust_remote_code=True
        )
        
        self.llm_pipe = pipeline(
            "text-generation",
            model=self.llm_model,
            tokenizer=self.llm_tokenizer,
            torch_dtype=torch.float16,
        )
        
        print(f"✅ LLM 로딩 완료 ({time.time() - start:.1f}초)")
    
    def load_all(self):
        """모든 모델 로딩"""
        self.load_faiss_index()
        self.load_embedding_model()
        self.load_llm()
    
    def get_query_embedding(self, query: str) -> np.ndarray:
        """쿼리 텍스트를 임베딩 벡터로 변환"""
        with torch.no_grad():
            inputs = self.embed_tokenizer(
                query,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(EMBEDDING_DEVICE)
            
            outputs = self.embed_model(**inputs)
            
            # Mean pooling
            attention_mask = inputs["attention_mask"]
            token_embeddings = outputs.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            
            # Normalize
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        
        return embeddings.cpu().numpy().astype('float32')
    
    def search_similar(self, query: str, top_k: int = TOP_K) -> list:
        """유사한 임상시험 검색"""
        start = time.time()
        
        # 쿼리 임베딩
        query_embedding = self.get_query_embedding(query)
        
        # FAISS 검색
        scores, indices = self.index.search(query_embedding, top_k)
        
        # 결과 구성
        results = []
        for i, idx in enumerate(indices[0]):
            nct_id = self.nct_ids[idx]
            study = self.study_data[nct_id]
            results.append({
                "nct_id": nct_id,
                "score": float(scores[0][i]),
                "data": study
            })
        
        print(f"✅ 유사 임상시험 검색 완료 ({time.time() - start:.2f}초)")
        for r in results:
            title = r["data"]["Study Overview"].get("Brief Title", "N/A")[:50]
            print(f"   - {r['nct_id']}: {title}... (유사도: {r['score']:.3f})")
        
        return results
    
    def build_rag_prompt(self, user_input: str, similar_studies: list) -> str:
        """Build RAG-based Few-shot CoT prompt in English"""
        
        system_prompt = """You are a clinical trial protocol expert.
    Generate a complete clinical trial protocol based on the user's research information.
    
    CRITICAL INSTRUCTION: ALL OUTPUT MUST BE IN ENGLISH, including:
    - Reasoning process
    - All JSON field values (Brief Title, Brief Summary, Conditions, etc.)
    - Even if the input is in Korean, translate and respond in English
    
    Follow these steps:
    1. Analyze the input and explain your reasoning in English
    2. Generate the protocol in JSON format with English content"""
    
        prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|>"
        
        # Few-shot 예시를 영어로 구성
        for i, study in enumerate(similar_studies, 1):
            nct_id = study["nct_id"]
            data = study["data"]
            overview = data.get("Study Overview", {})
            
            # 예시 입력도 영어로!
            example_input = f"""
    Reference Clinical Trial #{i} ({nct_id}):
    - Title: {overview.get('Brief Title', 'N/A')}
    - Conditions: {', '.join(overview.get('Conditions', []))}
    - Study Type: {overview.get('Study Type', 'N/A')}
    - Interventions: {', '.join([inv.get('Name', '') for inv in overview.get('Interventions', [])])}
    """
            
            example_output = json.dumps(data, ensure_ascii=False, indent=2)
            reasoning = self._generate_reasoning(data)
            
            # "Final Output"도 영어로
            prompt += f"""
    <|start_header_id|>user<|end_header_id|>
    
    {example_input}<|eot_id|>
    
    <|start_header_id|>assistant<|end_header_id|>
    
    {reasoning}
    
    Final Output:
    ````json
    {example_output}
    ```<|eot_id|>
    """
        
        # 사용자 입력 후 영어 응답 유도
        prompt += f"""
    <|start_header_id|>user<|end_header_id|>
    
    {user_input}
    
    (Note: Respond entirely in English)<|eot_id|>
    
    <|start_header_id|>assistant<|end_header_id|>
    
    Reasoning Process:
    1. Study Overview Analysis:
    """
        
        return prompt
    
    def _generate_reasoning(self, data: dict) -> str:
        """Generate English CoT reasoning from data"""
        overview = data.get("Study Overview", {})
        criteria = data.get("Participation Criteria", {})
        plan = data.get("Study Plan", {})
        
        study_type = overview.get("Study Type", "N/A")
        phases = overview.get("Phases", [])
        allocation = plan.get("Design Information", {}).get("Allocation", "N/A")
        masking = plan.get("Design Information", {}).get("Masking", "N/A")
        purpose = plan.get("Design Information", {}).get("Primary Purpose", "N/A")
        
        reasoning = f"""Reasoning Process:
    1. Study Overview Analysis:
       - Study Type: {study_type}
       - Phases: {', '.join(phases) if phases else 'N/A'}
       - Conditions: {', '.join(overview.get('Conditions', []))}
    
    2. Participation Criteria Analysis:
       - Healthy Volunteers: {criteria.get('Healthy Volunteers', 'N/A')}
       - Sex: {criteria.get('Sex', 'N/A')}
       - Age: {criteria.get('Minimum Age', 'N/A')} ~ {criteria.get('Maximum Age', 'N/A')}
    
    3. Study Plan Analysis:
       - Allocation: {allocation}
       - Masking: {masking}
       - Primary Purpose: {purpose}
    
    Final Output:"""
        
        return reasoning
    
    def generate_protocol(self, user_input: str, max_new_tokens: int = 2048) -> dict:
        """RAG 기반 프로토콜 생성"""

        set_all_seeds(SEED)
        
        # 1. 유사 임상시험 검색
        print("\n📍 Step 1: 유사 임상시험 검색")
        similar_studies = self.search_similar(user_input, TOP_K)
        
        # 2. RAG 프롬프트 구성
        print("\n📍 Step 2: 프롬프트 구성")
        prompt = self.build_rag_prompt(user_input, similar_studies)
        
        # 3. LLM 생성
        print("\n📍 Step 3: 프로토콜 생성 중...")
        start = time.time()
        
        outputs = self.llm_pipe(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            return_full_text=False
        )
        
        generated_text = outputs[0]["generated_text"]
        exe_time=(time.time() - start)
        
        if exe_time>=60:
            print(f"✅ 생성 완료 ({exe_time/60}분, {(exe_time%60)})")
        else:
            print(f"✅ 생성 완료 ({exe_time}초)")
        
        try:
            # Extract reasoning (before JSON block)
            json_start = generated_text.find("```json")
            if json_start != -1:
                reasoning_text = generated_text[:json_start].strip()
                json_end = generated_text.find("```", json_start + 7)
                if json_end != -1:
                    json_str = generated_text[json_start + 7:json_end].strip()
                    result = json.loads(json_str)
            else:
                # Try to find JSON without code block
                json_start = generated_text.find("{")
                if json_start != -1:
                    reasoning_text = generated_text[:json_start].strip()
                    json_end = generated_text.rfind("}") + 1
                    json_str = generated_text[json_start:json_end]
                    result = json.loads(json_str)
                
        except json.JSONDecodeError:
            print("⚠️ JSON parsing failed")
            result = {"raw_output": generated_text}
        
        return result, generated_text, similar_studies, reasoning_text
    
    
    def release_gpu(self):
        """GPU 메모리 해제"""
        if self.embed_model is not None:
            self.embed_model.cpu()
            del self.embed_model
        if self.embed_tokenizer is not None:
            del self.embed_tokenizer
        if self.llm_model is not None:
            self.llm_model.cpu()
            del self.llm_model
        if self.llm_pipe is not None:
            del self.llm_pipe
        if self.llm_tokenizer is not None:
            del self.llm_tokenizer
        
        import gc
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        print("✅ GPU 메모리 해제 완료")


# ============================================================
# 3. 메인 실행
# ============================================================

if __name__ == "__main__":
    rag = ClinicalTrialRAG()
    
    try:
        # 모델 로딩
        rag.load_all()
        
        # 사용자 입력
        user_input = """
사용자 입력 정보:
- 연구 주제: 폐색전증 의심 환자에서 초음파 검사의 진단적 유용성
- 연구 목적: CT 또는 폐환기관류스캔 의뢰 감소
- 대상: 응급실 내원 성인 환자
- 중재: Point-of-care 초음파 검사
- 기간: 24시간 내 결과 확인
"""
        
        print("\n" + "="*70)
        print("📋 RAG 기반 임상시험 프로토콜 생성")
        print("="*70)
        print(f"\n입력:\n{user_input}")
        
        # 프로토콜 생성
        result, raw_output, similar, reasoning = rag.generate_protocol(user_input)
         
        # Print reasoning process
        print("\n" + "="*70)
        print("🧠 Reasoning Process (CoT)")
        print("="*70)
        print(reasoning if reasoning else "No explicit reasoning found in output")
        
        # Print generated result
        print("\n" + "="*70)
        print("✅ Generated Protocol")
        print("="*70)
        print(json.dumps(result, ensure_ascii=False, indent=2))

        
        # 결과 저장
        os.makedirs("Results", exist_ok=True)
        
        output_data = {
            "input": user_input,
            "similar_studies": [{"nct_id": s["nct_id"], "score": s["score"]} for s in similar],
            "reasoning": reasoning,
            "generated_protocol": result,
            "raw_output": raw_output
        }
        
        with open("Results/rag_generated_protocol.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        
        print("\n💾 결과가 'Results/rag_generated_protocol.json'에 저장되었습니다.")
        
    finally:
        rag.release_gpu()